# LiDAR Wind Field Forecasting

Parametric study of Taylor's Frozen Turbulence Hypothesis (TFH) validity for LiDAR-based wind field forecasting.

**Research question:** How does prediction error of a TFH-based wind forecasting method vary as a function of measurement distance and turbulence intensity, and under which conditions does the hypothesis become unreliable for real-time wind turbine control?

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import signal as scipy_signal

os.makedirs('results', exist_ok=True)

# Simulation parameters
dt = 0.5          # time step (s)
T = 600           # total duration (s)
U = 10.0          # mean wind speed (m/s)
z = 80.0          # measurement height (m)

# Derived
N = int(T / dt)
t = np.arange(N) * dt
freqs = np.fft.rfftfreq(N, d=dt)
freqs[0] = 1e-6  # avoid division by zero

## 1. Synthetic Wind Generation (Kaimal Spectrum)

Wind time series are generated using the Kaimal spectral model (IEC 61400-1), which describes the energy distribution of atmospheric turbulence across frequencies.

In [ ]:
def kaimal_spectrum(f, U, z):
    """Kaimal spectral model for longitudinal wind turbulence."""
    L = 8.1 * 42        # integral length scale (m)
    n = f * L / U        # normalised frequency
    S = (4 * L / U) / (1 + 6 * n)**(5/3)
    return S

# Parameterised version with adjustable length scale
def kaimal_spectrum_L(f, U, L):
    """Kaimal spectral model with explicit length scale parameter."""
    n = f * L / U
    S = (4 * L / U) / (1 + 6 * n)**(5/3)
    return S

In [ ]:
# Generate synthetic wind time series
S = kaimal_spectrum(freqs, U, z)
df = freqs[1] - freqs[0]
amplitudes = np.sqrt(2 * S * df)

rng = np.random.default_rng(seed=42)
phases = rng.uniform(0, 2*np.pi, len(freqs))
fourier = amplitudes * np.exp(1j * phases)

u_fluct = np.real(np.fft.irfft(fourier, n=N)) * N
u = U + u_fluct

# Plot
plt.figure(figsize=(12, 4))
plt.plot(t, u, color='steelblue', linewidth=0.8)
plt.xlabel('Time (s)')
plt.ylabel('Wind speed (m/s)')
plt.title('Synthetic wind speed time series (Kaimal spectrum)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean wind speed:      {np.mean(u):.2f} m/s")
print(f"Standard deviation:   {np.std(u):.2f} m/s")
print(f"Turbulence intensity: {np.std(u)/np.mean(u)*100:.1f}%")

## 2. Taylor's Frozen Turbulence Hypothesis — Basic Test

TFH assumes that turbulent structures are advected downstream by the mean wind without evolving. The downstream signal is predicted by simply time-shifting the upstream measurement.

In [ ]:
# Taylor's hypothesis parameters
x_sep = 100.0              # separation distance (m)
tau = x_sep / U            # time lag (s)
lag_steps = int(tau / dt)  # time lag in samples

# Apply TFH: predicted downstream = time-shifted upstream
u_upstream = u[:-lag_steps]
u_downstream_actual = u[lag_steps:]
u_downstream_predicted = u_upstream
t_valid = t[lag_steps:]

# Error metrics
error = u_downstream_actual - u_downstream_predicted
rmse = np.sqrt(np.mean(error**2))
correlation = np.corrcoef(u_downstream_actual, u_downstream_predicted)[0, 1]

print(f"Separation distance: {x_sep} m")
print(f"Time lag:            {tau:.1f} s ({lag_steps} steps)")
print(f"RMSE:                {rmse:.4f} m/s")
print(f"Correlation:         {correlation:.4f}")

# Plot
plt.figure(figsize=(12, 5))
plt.plot(t_valid[:200], u_downstream_actual[:200],
         label='Actual', color='steelblue', linewidth=1)
plt.plot(t_valid[:200], u_downstream_predicted[:200],
         label='Taylor prediction', color='tomato', linewidth=1, linestyle='--')
plt.xlabel('Time (s)')
plt.ylabel('Wind speed (m/s)')
plt.title(f"Taylor's Hypothesis: Actual vs Predicted  |  RMSE={rmse:.3f} m/s  |  r={correlation:.3f}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Atmospheric Stability Comparison

Three atmospheric stability regimes are simulated by varying turbulence intensity and integral length scale:
- **Stable** (night-time, stratified flow): TI ≈ 5%
- **Neutral** (overcast, moderate mixing): TI ≈ 15%
- **Unstable** (daytime, strong convection): TI ≈ 30%

In [ ]:
stability_conditions = {
    'Stable':   {'L_scale': 8.1 * 120, 'sigma_u': 0.5},
    'Neutral':  {'L_scale': 8.1 * 42,  'sigma_u': 1.5},
    'Unstable': {'L_scale': 8.1 * 10,  'sigma_u': 3.0},
}

results = {}

for condition, params in stability_conditions.items():
    S = kaimal_spectrum_L(freqs, U, params['L_scale'])
    df = freqs[1] - freqs[0]
    amplitudes = np.sqrt(2 * S * df)

    rng = np.random.default_rng(seed=42)
    phases = rng.uniform(0, 2*np.pi, len(freqs))
    fourier = amplitudes * np.exp(1j * phases)
    u_fluct = np.real(np.fft.irfft(fourier, n=N)) * N
    u_fluct = u_fluct / np.std(u_fluct) * params['sigma_u']
    u_cond = U + u_fluct

    # Apply TFH
    u_up = u_cond[:-lag_steps]
    u_down_actual = u_cond[lag_steps:]
    u_down_predicted = u_up

    error = u_down_actual - u_down_predicted
    rmse = np.sqrt(np.mean(error**2))
    corr = np.corrcoef(u_down_actual, u_down_predicted)[0, 1]
    ti = np.std(u_cond) / np.mean(u_cond) * 100

    results[condition] = {
        'rmse': rmse, 'correlation': corr, 'turbulence_intensity': ti,
        'u': u_cond, 'u_actual': u_down_actual, 'u_predicted': u_down_predicted
    }
    print(f"{condition:10s} | TI={ti:.1f}% | RMSE={rmse:.3f} m/s | r={corr:.3f}")

In [ ]:
# Stability comparison — time series
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

for idx, (condition, data) in enumerate(results.items()):
    axes[idx].plot(t_valid[:200], data['u_actual'][:200],
                   color='steelblue', linewidth=1, label='Actual')
    axes[idx].plot(t_valid[:200], data['u_predicted'][:200],
                   color='tomato', linewidth=1, linestyle='--', label='Predicted')
    axes[idx].set_ylabel('Wind speed (m/s)')
    axes[idx].set_title(f"{condition} | TI={data['turbulence_intensity']:.1f}% | "
                        f"RMSE={data['rmse']:.3f} m/s | r={data['correlation']:.3f}")
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend(loc='upper right')

axes[-1].set_xlabel('Time (s)')
plt.suptitle("Taylor's Frozen Turbulence Hypothesis under Different Atmospheric Stability Conditions",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/stability_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table
print(f"{'Condition':<12} {'TI (%)':>8} {'RMSE (m/s)':>12} {'Correlation':>12} {'Validity'}")
print("-" * 55)
for condition, data in results.items():
    ti = data['turbulence_intensity']
    rmse = data['rmse']
    corr = data['correlation']
    validity = "Good" if corr > 0.6 else "Moderate" if corr > 0.4 else "Poor"
    print(f"{condition:<12} {ti:>8.1f} {rmse:>12.3f} {corr:>12.3f} {validity}")

In [ ]:
# Correlation and RMSE vs turbulence intensity
ti_values = [results[c]['turbulence_intensity'] for c in results]
corr_values = [results[c]['correlation'] for c in results]
rmse_values = [results[c]['rmse'] for c in results]
labels = list(results.keys())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ti_values, corr_values, 'o-', color='steelblue', linewidth=2, markersize=10)
for i, label in enumerate(labels):
    ax1.annotate(label, (ti_values[i], corr_values[i]),
                textcoords="offset points", xytext=(8, 4), fontsize=10)
ax1.set_xlabel('Turbulence Intensity (%)')
ax1.set_ylabel('Correlation (r)')
ax1.set_title("Taylor's Hypothesis Validity vs Turbulence Intensity")
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

ax2.plot(ti_values, rmse_values, 'o-', color='tomato', linewidth=2, markersize=10)
for i, label in enumerate(labels):
    ax2.annotate(label, (ti_values[i], rmse_values[i]),
                textcoords="offset points", xytext=(8, 4), fontsize=10)
ax2.set_xlabel('Turbulence Intensity (%)')
ax2.set_ylabel('RMSE (m/s)')
ax2.set_title('Prediction Error vs Turbulence Intensity')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/correlation_vs_ti.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Power spectral density under each stability condition
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {'Stable': 'green', 'Neutral': 'steelblue', 'Unstable': 'tomato'}

for idx, (condition, data) in enumerate(results.items()):
    f_psd, psd = scipy_signal.welch(data['u'], fs=1/dt, nperseg=256)
    axes[idx].loglog(f_psd[1:], psd[1:], color=colors[condition], linewidth=1.5)
    axes[idx].set_xlabel('Frequency (Hz)')
    axes[idx].set_ylabel('PSD (m²/s)')
    axes[idx].set_title(f'{condition} | TI={data["turbulence_intensity"]:.1f}%')
    axes[idx].grid(True, alpha=0.3, which='both')

plt.suptitle('Power Spectral Density under Different Stability Conditions',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('results/power_spectra.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Davenport Coherence Model

TFH assumes perfect coherence at all frequencies. In reality, coherence decays with frequency according to the Davenport model:

    γ(f) = exp(−C × f × Δx / U)

where C is the decay coefficient (8.0 for neutral conditions), f is frequency, Δx is separation distance, and U is mean wind speed. This captures the physical decorrelation of small-scale turbulent structures during advection.

In [ ]:
def davenport_coherence(f, x_sep, U, decay=8.0):
    """Frequency-dependent coherence between upstream and downstream signals.

    Parameters
    ----------
    f     : array — frequency (Hz)
    x_sep : float — separation distance (m)
    U     : float — mean wind speed (m/s)
    decay : float — Davenport decay coefficient (default 8.0)

    Returns
    -------
    gamma : array — coherence (0 to 1)
    """
    return np.exp(-decay * f * x_sep / U)

# Visualise coherence decay
gamma = davenport_coherence(freqs[1:], x_sep, U)

plt.figure(figsize=(10, 4))
plt.semilogx(freqs[1:], gamma, color='steelblue', linewidth=2)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Coherence γ')
plt.title(f'Davenport Coherence Function — x = {x_sep} m, U = {U} m/s')
plt.grid(True, alpha=0.3, which='both')
plt.axhline(y=0.5, color='tomato', linestyle='--', alpha=0.7, label='γ = 0.5')
plt.legend()
plt.tight_layout()
plt.show()

for f_val in [0.01, 0.1, 0.5, 1.0]:
    print(f"Coherence at {f_val:.2f} Hz: {davenport_coherence(f_val, x_sep, U):.3f}")

In [ ]:
def generate_downstream_signal(fourier_upstream, freqs, x_sep, U, sigma_u, seed=123):
    """Generate physically realistic downstream signal using Davenport coherence.

    Combines a coherent component (from upstream) and an incoherent component
    (independent noise) weighted by frequency-dependent coherence.
    """
    N_f = len(fourier_upstream)
    gamma = davenport_coherence(freqs, x_sep, U)

    # Independent noise with same spectral shape
    rng = np.random.default_rng(seed=seed)
    phases_noise = rng.uniform(0, 2*np.pi, N_f)
    S = kaimal_spectrum_L(freqs, U, 8.1 * 42)
    df = freqs[1] - freqs[0]
    amplitudes = np.sqrt(2 * S * df)
    fourier_noise = amplitudes * np.exp(1j * phases_noise)

    # Downstream = coherent part + incoherent part
    fourier_downstream = gamma * fourier_upstream + np.sqrt(1 - gamma**2) * fourier_noise
    n_time = len(fourier_upstream) * 2 - 2
    u_downstream = np.real(np.fft.irfft(fourier_downstream, n=n_time)) * n_time
    u_downstream = u_downstream / np.std(u_downstream) * sigma_u

    return U + u_downstream

In [ ]:
# Compare Taylor's prediction against Davenport-based downstream signal
S_test = kaimal_spectrum_L(freqs, U, 8.1 * 42)
df = freqs[1] - freqs[0]
amplitudes_test = np.sqrt(2 * S_test * df)
rng_test = np.random.default_rng(seed=42)
phases_test = rng_test.uniform(0, 2*np.pi, len(freqs))
fourier_upstream = amplitudes_test * np.exp(1j * phases_test)

u_downstream_realistic = generate_downstream_signal(fourier_upstream, freqs, x_sep, U, sigma_u=1.5)
u_upstream_signal = U + np.real(np.fft.irfft(fourier_upstream, n=N)) * N
u_upstream_signal = U + (u_upstream_signal - U) / np.std(u_upstream_signal - U) * 1.5

# Taylor prediction
u_taylor_predicted = u_upstream_signal[:-lag_steps]
u_downstream_actual = u_downstream_realistic[lag_steps:]
t_valid = t[lag_steps:]

error = u_downstream_actual - u_taylor_predicted
rmse_dav = np.sqrt(np.mean(error**2))
corr_dav = np.corrcoef(u_downstream_actual, u_taylor_predicted)[0, 1]

print(f"With Davenport coherence model (neutral, x={x_sep}m):")
print(f"  RMSE:        {rmse_dav:.4f} m/s")
print(f"  Correlation: {corr_dav:.4f}")

plt.figure(figsize=(12, 5))
plt.plot(t_valid[:300], u_downstream_actual[:300],
         color='steelblue', linewidth=1, label='Actual (Davenport)')
plt.plot(t_valid[:300], u_taylor_predicted[:300],
         color='tomato', linewidth=1, linestyle='--', label='Taylor prediction')
plt.xlabel('Time (s)')
plt.ylabel('Wind speed (m/s)')
plt.title(f'Taylor Prediction vs Davenport Downstream | RMSE={rmse_dav:.3f} m/s | r={corr_dav:.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Parameter Sweep — Distance × Turbulence Intensity

Full parameter sweep over 5 turbulence intensities × 5 separation distances × 20 realisations per case (500 total simulations). Each realisation uses independent random seeds for statistical stability.

In [ ]:
separation_distances = [50, 100, 150, 200, 300]
TI_values = [0.05, 0.10, 0.15, 0.20, 0.25]
n_realisations = 20

rmse_matrix = np.zeros((len(TI_values), len(separation_distances)))
corr_matrix = np.zeros((len(TI_values), len(separation_distances)))

print("Running parameter sweep with 20 realisations per case...")
print(f"{'TI':>6} {'x=50m':>8} {'x=100m':>8} {'x=150m':>8} {'x=200m':>8} {'x=300m':>8}")
print("-" * 50)

for i, ti in enumerate(TI_values):
    sigma = U * ti
    row_corr = []

    for j, x in enumerate(separation_distances):
        lag = int((x / U) / dt)
        if lag == 0 or lag >= N:
            row_corr.append('N/A')
            continue

        corr_runs = []
        rmse_runs = []

        for k in range(n_realisations):
            seed_up = i * 1000 + j * 100 + k
            seed_dn = i * 1000 + j * 100 + k + 500

            # Generate upstream
            S = kaimal_spectrum_L(freqs, U, 8.1 * 42)
            df = freqs[1] - freqs[0]
            amplitudes = np.sqrt(2 * S * df)

            rng = np.random.default_rng(seed=seed_up)
            phases = rng.uniform(0, 2*np.pi, len(freqs))
            fourier_up = amplitudes * np.exp(1j * phases)
            u_fluct = np.real(np.fft.irfft(fourier_up, n=N)) * N
            u_fluct = u_fluct / np.std(u_fluct) * sigma
            u_up = U + u_fluct
            fourier_up_scaled = np.fft.rfft(u_fluct) / N

            # Generate downstream with Davenport coherence
            gamma = davenport_coherence(freqs, x, U)
            rng2 = np.random.default_rng(seed=seed_dn)
            phases_noise = rng2.uniform(0, 2*np.pi, len(freqs))
            fourier_noise = amplitudes * np.exp(1j * phases_noise)
            u_noise = np.real(np.fft.irfft(fourier_noise, n=N)) * N
            u_noise = u_noise / np.std(u_noise) * sigma
            fourier_noise_scaled = np.fft.rfft(u_noise) / N

            fourier_down = (gamma * fourier_up_scaled +
                          np.sqrt(np.maximum(1 - gamma**2, 0)) * fourier_noise_scaled)
            u_down_fluct = np.real(np.fft.irfft(fourier_down, n=N)) * N
            u_down_fluct = u_down_fluct / np.std(u_down_fluct) * sigma
            u_down = U + u_down_fluct

            # Taylor prediction
            u_pred = u_up[:-lag]
            u_act = u_down[lag:]
            min_len = min(len(u_pred), len(u_act))

            error = u_act[:min_len] - u_pred[:min_len]
            rmse_runs.append(np.sqrt(np.mean(error**2)))
            corr_runs.append(np.corrcoef(u_act[:min_len], u_pred[:min_len])[0, 1])

        rmse_matrix[i, j] = np.mean(rmse_runs)
        corr_matrix[i, j] = np.mean(corr_runs)
        row_corr.append(f"{corr_matrix[i, j]:.3f}")

    print(f"{ti*100:>5.0f}% {' '.join(f'{v:>8}' for v in row_corr)}")

print("\nParameter sweep complete — averaged over 20 realisations.")

In [ ]:
# Heatmaps — main result figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

im1 = ax1.imshow(corr_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=0.7)
ax1.set_xticks(range(len(separation_distances)))
ax1.set_xticklabels([f'{x}m' for x in separation_distances])
ax1.set_yticks(range(len(TI_values)))
ax1.set_yticklabels([f'{ti*100:.0f}%' for ti in TI_values])
ax1.set_xlabel('Separation Distance')
ax1.set_ylabel('Turbulence Intensity')
ax1.set_title("Correlation (r) — Taylor's Hypothesis Validity")
plt.colorbar(im1, ax=ax1)
for ii in range(len(TI_values)):
    for jj in range(len(separation_distances)):
        ax1.text(jj, ii, f'{corr_matrix[ii, jj]:.2f}',
                ha='center', va='center', fontsize=11, fontweight='bold')

im2 = ax2.imshow(rmse_matrix, cmap='RdYlGn_r', aspect='auto')
ax2.set_xticks(range(len(separation_distances)))
ax2.set_xticklabels([f'{x}m' for x in separation_distances])
ax2.set_yticks(range(len(TI_values)))
ax2.set_yticklabels([f'{ti*100:.0f}%' for ti in TI_values])
ax2.set_xlabel('Separation Distance')
ax2.set_ylabel('Turbulence Intensity')
ax2.set_title('RMSE (m/s) — Prediction Error')
plt.colorbar(im2, ax=ax2)
for ii in range(len(TI_values)):
    for jj in range(len(separation_distances)):
        ax2.text(jj, ii, f'{rmse_matrix[ii, jj]:.2f}',
                ha='center', va='center', fontsize=11, fontweight='bold')

plt.suptitle("Taylor's Frozen Turbulence Hypothesis — Parameter Space Analysis\n"
             "Davenport Coherence Model | 20 realisations per case",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/parameter_sweep_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Real Data Validation

Two publicly available wind farm datasets are used for consistency checks against the synthetic analysis.

### 6.1 PSD Validation — RW-Turb Dataset

100 Hz sonic anemometer data from the Pays d'Othe wind farm (France) at two heights (45m and 80m). The power spectral density is compared against the Kaimal model to verify the -5/3 inertial subrange slope.

**Data source:** Gires et al. (2022), Zenodo. DOI: [10.5281/zenodo.5801900](https://doi.org/10.5281/zenodo.5801900)

In [ ]:
# Load 100Hz sonic anemometer data
# NOTE: Download from https://doi.org/10.5281/zenodo.5801900 and place in data/
import pandas as pd

path_anem1 = 'data/Hourly_data_100Hz_Anemometer_1_20210301_00.csv'
path_anem2 = 'data/Hourly_data_100Hz_Anemometer_2_20210301_00.csv'

df1 = pd.read_csv(path_anem1, sep=';', header=None, names=['u', 'v', 'w', 'T'])
df2 = pd.read_csv(path_anem2, sep=';', header=None, names=['u', 'v', 'w', 'T'])

fs = 100  # Hz

# Horizontal wind speed at both heights
ws1 = np.sqrt(df1['u'].astype(float)**2 + df1['v'].astype(float)**2).dropna().values
ws2 = np.sqrt(df2['u'].astype(float)**2 + df2['v'].astype(float)**2).dropna().values

U1, U2 = np.mean(ws1), np.mean(ws2)
sigma1, sigma2 = np.std(ws1), np.std(ws2)
TI1, TI2 = sigma1 / U1, sigma2 / U2

print(f"Anemometer 1 (45m): U={U1:.2f} m/s, TI={TI1*100:.1f}%")
print(f"Anemometer 2 (80m): U={U2:.2f} m/s, TI={TI2*100:.1f}%")
print(f"Duration: {len(ws1)/fs:.0f} s ({len(ws1)/fs/60:.1f} min)")

In [ ]:
# PSD comparison — real data vs Kaimal model
freqs_w1, psd_w1 = scipy_signal.welch(ws1 - U1, fs=fs, nperseg=4096, window='hann')
freqs_w2, psd_w2 = scipy_signal.welch(ws2 - U2, fs=fs, nperseg=4096, window='hann')

L = 8.1 * 42
S_kaimal_1 = kaimal_spectrum_L(freqs_w1[1:], U1, L)
S_kaimal_2 = kaimal_spectrum_L(freqs_w2[1:], U2, L)

# Normalise for shape comparison
psd_w1_norm = psd_w1 / sigma1**2
psd_w2_norm = psd_w2 / sigma2**2
S_kaimal_1_norm = S_kaimal_1 / sigma1**2
S_kaimal_2_norm = S_kaimal_2 / sigma2**2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.loglog(freqs_w1[1:], psd_w1_norm[1:], color='steelblue', linewidth=1.5,
           alpha=0.8, label='Real data (45m)')
ax1.loglog(freqs_w1[1:], S_kaimal_1_norm, color='tomato',
           linewidth=2, linestyle='--', label='Kaimal model')
f_ref = freqs_w1[10:100]
ax1.loglog(f_ref, 0.01 * f_ref**(-5/3), color='green',
           linewidth=1.5, linestyle=':', label='-5/3 slope (Kolmogorov)')
ax1.set_xlabel('Frequency (Hz)')
ax1.set_ylabel('Normalised PSD (1/Hz)')
ax1.set_title(f'Anemometer 1 — 45m | U={U1:.1f} m/s, TI={TI1*100:.1f}%')
ax1.legend()
ax1.grid(True, alpha=0.3, which='both')
ax1.set_xlim([0.01, 10])

ax2.loglog(freqs_w2[1:], psd_w2_norm[1:], color='steelblue', linewidth=1.5,
           alpha=0.8, label='Real data (80m)')
ax2.loglog(freqs_w2[1:], S_kaimal_2_norm, color='tomato',
           linewidth=2, linestyle='--', label='Kaimal model')
ax2.loglog(f_ref, 0.01 * f_ref**(-5/3), color='green',
           linewidth=1.5, linestyle=':', label='-5/3 slope (Kolmogorov)')
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Normalised PSD (1/Hz)')
ax2.set_title(f'Anemometer 2 — 80m | U={U2:.1f} m/s, TI={TI2*100:.1f}%')
ax2.legend()
ax2.grid(True, alpha=0.3, which='both')
ax2.set_xlim([0.01, 10])

plt.suptitle("Normalised PSD — Real 100Hz Wind Farm Data vs Kaimal Model\n"
             "RW-Turb Dataset, Pays d'Othe Wind Farm, France, March 2021",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('results/real_data_100Hz_psd_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Longitudinal Coherence — UEBB Dataset

Concurrent LiDAR and met mast measurements from the Beberibe wind farm (Brazil). Coherence is computed between met mast wind speed at 60m and LiDAR measurements at range gates from 50–300m, and compared against the Davenport model.

**Data source:** Passos et al. (2017), Zenodo. DOI: [10.5281/zenodo.1475197](https://doi.org/10.5281/zenodo.1475197)

In [ ]:
# Load UEBB dataset
# NOTE: Download from https://doi.org/10.5281/zenodo.1475197 and place in data/
import xarray as xr

ds = xr.open_dataset('data/UEPS_v1.nc')

# Extract LiDAR and met mast data
lidar_ranges = ds['range'].values
ws_mast = ds['wind_speed'].sel(Height=60.0).values
ws_lidar = ds['lidar_wind_speed'].values

U_uebb = np.nanmean(ws_mast)
print(f"LiDAR range gates (m): {lidar_ranges}")
print(f"Mean wind speed: {U_uebb:.2f} m/s")

# Data availability per range gate
print("\nLiDAR availability per range gate:")
for i, r in enumerate(lidar_ranges[:10]):
    nan_pct = 100 * np.sum(np.isnan(ws_lidar[i, :])) / ws_lidar.shape[1]
    print(f"  Range {r:.0f}m: {100-nan_pct:.1f}% available")

In [ ]:
# Longitudinal coherence — LiDAR vs met mast vs Davenport model
fs_uebb = 1/600  # 10-min intervals

target_ranges = [50, 100, 150, 200, 300]
range_indices = [np.argmin(np.abs(lidar_ranges - r)) for r in target_ranges]
actual_ranges = [lidar_ranges[i] for i in range_indices]

fig, axes = plt.subplots(1, 5, figsize=(18, 5), sharey=True)

for idx, (r_target, r_actual, r_idx) in enumerate(zip(target_ranges, actual_ranges, range_indices)):
    # Get concurrent valid data
    ws_lidar_range = ws_lidar[r_idx, :]
    valid = ~np.isnan(ws_mast) & ~np.isnan(ws_lidar_range)
    ws_m = ws_mast[valid]
    ws_l = ws_lidar_range[valid]

    # Compute coherence
    freqs_c, coh = scipy_signal.coherence(ws_l, ws_m, fs=fs_uebb, nperseg=256, window='hann')

    # Davenport model
    gamma_dav = davenport_coherence(freqs_c[1:], r_actual, U_uebb, decay=8.0)**2

    ax = axes[idx]
    ax.semilogx(freqs_c[1:], coh[1:], color='steelblue', linewidth=2,
                alpha=0.8, label='Real data')
    ax.semilogx(freqs_c[1:], gamma_dav, color='tomato', linewidth=2,
                linestyle='--', label='Davenport')
    ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.7)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_title(f'x = {r_actual:.0f}m')
    ax.grid(True, alpha=0.3, which='both')
    ax.set_ylim([0, 1.1])
    if idx == 0:
        ax.set_ylabel('Coherence γ²')
        ax.legend(fontsize=8)

plt.suptitle("Longitudinal Coherence — LiDAR vs Met Mast | UEBB Dataset\n"
             "Real measurements vs Davenport coherence model",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('results/real_data_lidar_coherence.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

**Key finding:** TFH correlation degrades from r ≈ 0.51 at 50m/5% TI to r ≈ 0.12 at 300m/20% TI. Distance dominates over turbulence intensity — TFH becomes unreliable beyond 100m separation regardless of TI.

**Real data validation** confirms:
- The Kaimal spectral model correctly reproduces the -5/3 inertial subrange slope observed in 100 Hz wind farm measurements
- The Davenport coherence model captures the frequency-dependent decorrelation pattern observed in LiDAR–met mast comparisons, though it overestimates coherence at higher frequencies under real coastal conditions

**Implications:** LiDAR-based feedforward control systems that rely on TFH should limit measurement distances to < 100m or incorporate coherence-based corrections for longer ranges, particularly under unstable atmospheric conditions.